# Notebook 05 – Dự báo chuỗi thời gian & Tổng kết

**Bài toán:** Dự báo nhiệt độ trung bình ngày.  

**Kỹ thuật:**
- Baseline 1: Naive (last value)
- Baseline 2: Moving Average (7 ngày)
- ARIMA(2,1,2): dựa trên phân tích ACF/PACF
- Holt-Winters (additive trend + seasonal, period=365): bắt seasonality năm

**Thiết lập:**
- Resample hourly → daily mean
- Split theo thời gian (80/20, KHÔNG shuffle)
- Metric: MAE, RMSE, sMAPE

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from src.data.loader import load_config, load_processed_data
from src.models.forecasting import TimeSeriesForecaster, check_stationarity
from src.evaluation.report import save_table
from src.visualization import plots

cfg = load_config('../configs/params.yaml')
plots.FIG_DIR = '../outputs/figures'
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

## 5.1 Chuẩn bị chuỗi thời gian

In [2]:
df = load_processed_data('../data/processed/weather_processed.parquet')
forecaster = TimeSeriesForecaster(cfg)
series = forecaster.prepare_series(df)
print(f'Chuỗi: {len(series)} ngày')
series.head(5)

[loader] Loaded processed data: (95150, 16)
[forecast] Series length: 4019 days, 2005-12-31 00:00:00 → 2016-12-31 00:00:00
Chuỗi: 4019 ngày


Formatted Date
2005-12-31    0.577778
2006-01-01    4.075000
2006-01-02    5.263194
2006-01-03    2.340509
2006-01-04    2.251932
Freq: D, Name: Temperature (C), dtype: float64

## 5.2 Kiểm tra tính dừng (ADF Test)

In [3]:
stat_result = check_stationarity(series)
for k, v in stat_result.items():
    print(f'  {k}: {v}')
if stat_result['is_stationary']:
    print('→ Chuỗi DỪNG, có thể dùng ARIMA trực tiếp')
else:
    print('→ Chuỗi KHÔNG dừng, cần differencing (d≥1)')

  adf_statistic: -3.9953
  p_value: 0.0014
  is_stationary: True
→ Chuỗi DỪNG, có thể dùng ARIMA trực tiếp


## 5.3 ACF/PACF → Chọn tham số ARIMA

In [4]:
plots.plot_acf_pacf(series)
print(f'→ Dựa vào ACF/PACF, chọn ARIMA{tuple(cfg["timeseries"]["arima_order"])}')

[plots] Saved: ../outputs/figures\ts_acf_pacf.png
→ Dựa vào ACF/PACF, chọn ARIMA(2, 1, 2)


## 5.4 Train/Test Split (theo thời gian)

In [5]:
train, test = forecaster.train_test_split_ts(series)
print(f'Train: {train.index[0]} → {train.index[-1]} ({len(train)} ngày)')
print(f'Test:  {test.index[0]} → {test.index[-1]} ({len(test)} ngày)')

[forecast] Train: 3215, Test: 804
Train: 2005-12-31 00:00:00 → 2014-10-19 00:00:00 (3215 ngày)
Test:  2014-10-20 00:00:00 → 2016-12-31 00:00:00 (804 ngày)


## 5.5 Train các mô hình

In [6]:
pred_naive = forecaster.naive_baseline(train, test)
pred_ma = forecaster.moving_average(train, test, window=7)
pred_arima = forecaster.fit_arima(train, test)
pred_hw = forecaster.fit_holtwinters(train, test)

  Naive: MAE=7.2108, RMSE=8.3510, sMAPE=69.55%
  MA(7): MAE=7.9459, RMSE=9.2553, sMAPE=70.17%
[forecast] Fitting ARIMA(2, 1, 2)...


  ARIMA: MAE=7.5674, RMSE=8.7028, sMAPE=69.43%
[forecast] Fitting Holt-Winters...


  HoltWinters: MAE=3.0962, RMSE=3.8362, sMAPE=42.73%


C:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


## 5.6 So sánh kết quả

In [7]:
results_ts = forecaster.get_results_table()
print(results_ts.to_string(index=False))
save_table(results_ts, 'ts_forecast_results', '../outputs/tables')

# So sánh tự động
best = results_ts.iloc[0]
worst = results_ts.iloc[-1]
print(f'\nMô hình tốt nhất: {best["Model"]} (MAE={best["MAE"]}°C)')
print(f'Mô hình kém nhất: {worst["Model"]} (MAE={worst["MAE"]}°C)')
print(f'Cải thiện MAE so baseline: {worst["MAE"] - best["MAE"]:.2f}°C')

      Model    MAE   RMSE  sMAPE(%)
HoltWinters 3.0962 3.8362   42.7299
      Naive 7.2108 8.3510   69.5545
      ARIMA 7.5674 8.7028   69.4261
      MA(7) 7.9459 9.2553   70.1669
[report] Saved table: ../outputs/tables\ts_forecast_results.csv

Mô hình tốt nhất: HoltWinters (MAE=3.0962°C)
Mô hình kém nhất: MA(7) (MAE=7.9459°C)
Cải thiện MAE so baseline: 4.85°C


In [8]:
forecasts = {'Naive': pred_naive, 'MA(7)': pred_ma, 'ARIMA': pred_arima, 'HoltWinters': pred_hw}
plots.plot_timeseries_forecast(train, test, forecasts, 'Dự báo Nhiệt độ (°C)')

[plots] Saved: ../outputs/figures\ts_forecast_comparison.png


## 5.7 Phân tích Residual

In [9]:
best_name = results_ts.iloc[0]['Model']
best_pred = forecasts[best_name]
plots.plot_residuals(test, best_pred, best_name)

[plots] Saved: ../outputs/figures\ts_residuals_holtwinters.png


In [10]:
# Residual theo tháng
import matplotlib.pyplot as plt
residuals = test.values - best_pred.values
residual_df = pd.DataFrame({
    'residual': residuals, 'month': test.index.month,
    'abs_residual': np.abs(residuals)
}, index=test.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
residual_df.boxplot(column='residual', by='month', ax=axes[0])
axes[0].set_title(f'Residual theo tháng – {best_name}')
axes[0].set_xlabel('Tháng')
axes[0].set_ylabel('Residual (°C)')
axes[0].axhline(0, color='red', linestyle='--')

monthly_mae = residual_df.groupby('month')['abs_residual'].mean()
axes[1].bar(monthly_mae.index, monthly_mae.values, color='#4C72B0', alpha=0.8)
axes[1].set_title(f'MAE theo tháng – {best_name}')
axes[1].set_xlabel('Tháng')
axes[1].set_ylabel('MAE (°C)')
axes[1].set_xticks(range(1, 13))
plt.tight_layout()
plt.savefig('../outputs/figures/ts_residual_by_month.png', bbox_inches='tight')
plt.close()

print(f'Tháng MAE cao nhất: {monthly_mae.idxmax()} ({monthly_mae.max():.2f}°C)')
print(f'Tháng MAE thấp nhất: {monthly_mae.idxmin()} ({monthly_mae.min():.2f}°C)')
save_table(monthly_mae.reset_index(), 'ts_mae_by_month', '../outputs/tables')

Tháng MAE cao nhất: 4 (4.14°C)
Tháng MAE thấp nhất: 12 (2.10°C)
[report] Saved table: ../outputs/tables\ts_mae_by_month.csv


'../outputs/tables\\ts_mae_by_month.csv'

## 5.8 Tổng kết dự án

Tóm tắt tự động từ kết quả pipeline.

In [11]:
print('=' * 65)
print('  TỔNG KẾT DỰ ÁN – ĐỀ TÀI 5: DỰ BÁO THỜI TIẾT')
print('=' * 65)

# 1. Load kết quả từ các bước trước
assoc_path = '../outputs/tables/association_top_rules.csv'
if os.path.exists(assoc_path):
    assoc_df = pd.read_csv(assoc_path)
    print(f'\n1. LUẬT KẾT HỢP: {len(assoc_df)} luật top, lift cao nhất = {assoc_df["lift"].max():.2f}')

cluster_path = '../outputs/tables/cluster_profile.csv'
if os.path.exists(cluster_path):
    cl_df = pd.read_csv(cluster_path)
    print(f'2. PHÂN CỤM: {len(cl_df)} cụm')
    if 'ClusterName' in cl_df.columns:
        print(f'   Tên cụm: {list(cl_df["ClusterName"])}')

clf_path = '../outputs/tables/clf_results_comparison.csv'
if os.path.exists(clf_path):
    clf_df = pd.read_csv(clf_path)
    best_clf = clf_df.sort_values('F1_macro', ascending=False).iloc[0]
    print(f'3. PHÂN LỚP: best={best_clf["Model"]} F1-macro={best_clf["F1_macro"]}')

print(f'4. CHUỖI THỜI GIAN: best={results_ts.iloc[0]["Model"]} MAE={results_ts.iloc[0]["MAE"]}°C')

insight_path = '../outputs/tables/actionable_insights.csv'
if os.path.exists(insight_path):
    ins_df = pd.read_csv(insight_path)
    print(f'5. INSIGHTS: {len(ins_df)} actionable insights')

print('\nHƯỚNG PHÁT TRIỂN:')
print('  - Thêm exogenous variables (Humidity, Pressure) vào SARIMAX')
print('  - Thử LSTM/Prophet cho time series non-linear')
print('  - Xây dựng ensemble theo mùa cho classification')
print('  - Tạo interaction features (Wind×Humidity, Temp-AppTemp)')
print('  - Bổ sung Streamlit demo app (điểm thưởng)')

# Liệt kê output files
print('\nFILES GENERATED:')
for folder in ['outputs/figures', 'outputs/tables', 'outputs/models']:
    path = os.path.join('..', folder)
    if os.path.exists(path):
        files = sorted(os.listdir(path))
        print(f'  {folder}/: {len(files)} files')

  TỔNG KẾT DỰ ÁN – ĐỀ TÀI 5: DỰ BÁO THỜI TIẾT

1. LUẬT KẾT HỢP: 20 luật top, lift cao nhất = 8.19
2. PHÂN CỤM: 4 cụm
   Tên cụm: ['Cold & Damp', 'Hot & Dry', 'Arctic Cold', 'Mild']
3. PHÂN LỚP: best=XGBoost F1-macro=0.7578
4. CHUỖI THỜI GIAN: best=HoltWinters MAE=3.0962°C
5. INSIGHTS: 6 actionable insights

HƯỚNG PHÁT TRIỂN:
  - Thêm exogenous variables (Humidity, Pressure) vào SARIMAX
  - Thử LSTM/Prophet cho time series non-linear
  - Xây dựng ensemble theo mùa cho classification
  - Tạo interaction features (Wind×Humidity, Temp-AppTemp)
  - Bổ sung Streamlit demo app (điểm thưởng)

FILES GENERATED:
  outputs/figures/: 29 files
  outputs/tables/: 22 files
  outputs/models/: 1 files
